In [ ]:
import pandas as pd

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
## load the data set
data = pd.read_csv("Churn_Modelling.csv")


In [ ]:
data.head(2)

In [ ]:
## delete the irrelevent data
data = data.drop(["RowNumber","CustomerId","Surname"] ,axis=1)

In [ ]:
data.head(2)

In [ ]:
## categorygies gender and geography
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# One-Hot Encode Gender

gender_onehot_encoder = OneHotEncoder(sparse_output=False)

gender_encoded = gender_onehot_encoder.fit_transform(data[['Gender']])

gender_encoded_df = pd.DataFrame( gender_encoded,columns=gender_onehot_encoder.get_feature_names_out(['Gender']))

In [ ]:
# Merge Encoded Columns
data = pd.concat(
    [
        data.drop('Gender', axis=1),
        gender_encoded_df
    ],
    axis=1
)

In [ ]:
joblib.dump(
    gender_onehot_encoder,
    'gender_encoder.pkl'
)


In [ ]:
# One-Hot Encode Geography

geo_onehot_encoder = OneHotEncoder(
    sparse_output=False
)

geo_encoded = geo_onehot_encoder.fit_transform(
    data[['Geography']]
)

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=geo_onehot_encoder.get_feature_names_out(
        ['Geography']
    )
)


data = pd.concat(
    [
        data.drop('Geography', axis=1),
        geo_encoded_df
    ],
    axis=1
)



joblib.dump(
    geo_onehot_encoder,
    'geo_encoder.pkl'
)

joblib.dump(
    scaler,
    'scaler.pkl'
)

print("Saved Successfully!")

In [ ]:
data.head(2)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
## Dicide the dataset into indepent and dependent features
x = data.drop('Exited',axis=1)
y = data['Exited']

## Split the data in training and testing sets

x_train , x_test , y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

## scale these features 

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
## save the scaler file
import joblib
joblib.dump(scaler,'scaler.pkl')    

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense  , Dropout
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
## build ann model 
model = Sequential([
    Dense(64, activation='relu', input_shape=(x_train.shape[1],)),
    Dropout(0.3),   # reduced
    Dense(32, activation='relu'),
    Dropout(0.2),   # reduced
    Dense(1, activation='sigmoid')
])

In [ ]:
# to check dropout
print(model.layers[1].rate)
print(model.layers[3].rate)

In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard


In [ ]:
## set the tensorboard
log_dir = "logs/fit" + datetime.datetime.now().strftime("%y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
tensorflow_callback

In [ ]:
## Set up early stop callback
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [ ]:
y_train.shape


In [ ]:
## Train the model
history = model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100 ,callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
model.save('ann_model.keras')

In [ ]:
## Load Tensorboard Extesion
%load_ext tensorboard

In [ ]:
## Launch tensorboard Session
%tensorboard --logdir logs/fit260628-154417